In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from bayes_opt import BayesianOptimization
import warnings
import random

warnings.filterwarnings('ignore')

random_state = 1
np.random.seed(random_state)
random.seed(random_state)

def fill_nan_with_median(data, name="数据"):
    """使用各列的中位数（数值列）或众数/默认值（非数值列）填充NaN"""
    if data.isnull().values.any():
        nan_count = data.isnull().sum().sum()
        print(f"警告：{name}包含 {nan_count} 个NaN值，将使用中位数/众数进行填充")
        
        num_cols = data.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            if data[col].isnull().any():
                median_val = data[col].median()
                if pd.isna(median_val): median_val = 0.0
                data[col] = data[col].fillna(median_val)
                
        non_num_cols = data.select_dtypes(exclude=[np.number]).columns
        for col in non_num_cols:
            if data[col].isnull().any():
                mode_val = data[col].mode()
                fill_val = mode_val[0] if not mode_val.empty else 'missing'
                data[col] = data[col].fillna(fill_val)
                
        print(f"{name}：共填充了 {nan_count} 个NaN值")
    return data

def CV_group(data):
    """负载均衡的数据分组策略，将数据按ID均匀分配到5个组"""
    data = data.copy()
    data = fill_nan_with_median(data, "CV_group输入数据")
    
    data['ref_FO'] = data['ID'].map(str)
    ref_JQ = data['ID']
    tt_ion = pd.DataFrame(ref_JQ.value_counts(ascending=False))
    tt_ion_index = tt_ion.index

    one, two, three, four, five = pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    for i in range(len(tt_ion_index)):
        a = [len(one), len(two), len(three), len(four), len(five)]
        min_idx = a.index(min(a))
        current_data = data[data['ID'] == tt_ion_index[i]]
        
        if min_idx == 0: one = pd.concat([one, current_data])
        elif min_idx == 1: two = pd.concat([two, current_data])
        elif min_idx == 2: three = pd.concat([three, current_data])
        elif min_idx == 3: four = pd.concat([four, current_data])
        elif min_idx == 4: five = pd.concat([five, current_data])

    print(f"分组长度: one={len(one)}, two={len(two)}, three={len(three)}, four={len(four)}, five={len(five)}")
    return one, two, three, four, five

class EnsembleModel:
    """封装 XGBoost 和 RandomForest 的加权融合模型，提供统一的 predict 接口"""
    def __init__(self, xgb_model, rf_model, scaler_X, weight_xgb):
        self.xgb_model = xgb_model
        self.rf_model = rf_model
        self.scaler_X = scaler_X
        self.weight_xgb = weight_xgb
        self.is_fitted_ = True 
        
    def predict(self, X):
        if isinstance(X, pd.Series):
            X = X.to_frame().T
        elif isinstance(X, np.ndarray) and X.ndim == 1:
            X = X.reshape(1, -1)
            
        if isinstance(X, pd.DataFrame):
            X = X.values
            
        X_scaled = self.scaler_X.transform(X)
        pred_xgb = self.xgb_model.predict(X_scaled)
        pred_rf = self.rf_model.predict(X)
        
        return self.weight_xgb * pred_xgb + (1 - self.weight_xgb) * pred_rf

class ML_model():
    def __init__(self, train, test, one, two, three, four, five, features):
        self.features = features
        self.scaler_X = StandardScaler()
        
        for name, df in [("训练集", train), ("测试集", test), ("分组1", one), ("分组2", two), 
                         ("分组3", three), ("分组4", four), ("分组5", five)]:
            self._check_and_fill_data(df, name)
        
        self.X_train = pd.DataFrame(train)
        self.X_test = pd.DataFrame(test)
        
        if 'Js' not in train.columns or 'Js' not in test.columns:
            raise ValueError("数据必须包含'Js'列作为目标变量")
        
        self.y_train = train['Js']
        self.y_test = test['Js']
        
        self.X_train_1 = pd.concat([two, three, four, five], join='outer')
        self.X_test_1 = one.copy()
        self.X_train_2 = pd.concat([one, three, four, five], join='outer')
        self.X_test_2 = two.copy()
        self.X_train_3 = pd.concat([one, two, four, five], join='outer')
        self.X_test_3 = three.copy()
        self.X_train_4 = pd.concat([one, two, three, five], join='outer')
        self.X_test_4 = four.copy()
        self.X_train_5 = pd.concat([one, two, three, four], join='outer')
        self.X_test_5 = five.copy()
        
        self._clean_cv_datasets()
        
        for i in range(1, 6):
            setattr(self, f'y_train_{i}', self._safe_get_target(getattr(self, f'X_train_{i}'), 'Js', f'分组{i}训练集'))
            setattr(self, f'y_test_{i}', self._safe_get_target(getattr(self, f'X_test_{i}'), 'Js', f'分组{i}测试集'))
            setattr(self, f'X_train_{i}', self._safe_select_features(getattr(self, f'X_train_{i}'), f"分组{i}训练集"))
            setattr(self, f'X_test_{i}', self._safe_select_features(getattr(self, f'X_test_{i}'), f"分组{i}测试集"))
            
        print("数据初始化完成，所有NaN问题已处理")
    
    def _check_and_fill_data(self, data, name):
        if data is None or len(data) == 0: return
        if data.isnull().sum().sum() > 0: fill_nan_with_median(data, name)
    
    def _clean_cv_datasets(self):
        for i in range(1, 6):
            for prefix, name in [('X_train', '训练集'), ('X_test', '测试集')]:
                df = getattr(self, f'{prefix}_{i}')
                if df is not None and df.isnull().sum().sum() > 0:
                    fill_nan_with_median(df, f"{name}{i}")
    
    def _safe_get_target(self, data, target_col, name):
        if target_col not in data.columns: raise ValueError(f"{name} 缺少目标列 '{target_col}'")
        y = data[target_col].copy()
        if y.isnull().any(): y = y.fillna(y.median())
        return y
    
    def _safe_select_features(self, data, name):
        X = data[self.features].copy()
        if X.isnull().values.any():
            for col in X.columns: X[col] = X[col].fillna(X[col].median())
        if np.isinf(X.values).any():
            X = X.replace([np.inf, -np.inf], np.nan)
            for col in X.columns: X[col] = X[col].fillna(X[col].median())
        return X
    
    def rem_name(self, org_data):
        if 'ref_ID' not in org_data.columns: return pd.DataFrame(index=org_data.index)
        return org_data.loc[:, ['ref_ID']].copy()
    
    def CV_train(self, params, X_train, y_train, X_test, y_test):
        """交叉验证训练方法 (XGBoost + RandomForest 融合版本)"""
        try:
            if X_train.empty or X_test.empty or len(y_train) == 0 or len(y_test) == 0:
                return float('inf')
            
            X_train_np = np.nan_to_num(X_train.values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            y_train_np = y_train.values.astype(np.float32)
            X_test_np = np.nan_to_num(X_test.values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            y_test_np = y_test.values.astype(np.float32)
            
            scaler_X = StandardScaler()
            X_train_scaled = scaler_X.fit_transform(X_train_np)
            X_test_scaled = scaler_X.transform(X_test_np)
            
            xgb_model = xgb.XGBRegressor(
                n_estimators=int(round(params['xgb_n_estimators'])),
                max_depth=int(round(params['xgb_max_depth'])),
                learning_rate=float(params['xgb_learning_rate']),
                subsample=float(params['xgb_subsample']),
                colsample_bytree=0.8,
                reg_alpha=0.1, reg_lambda=1.0,
                random_state=random_state, n_jobs=-1, verbosity=0
            )
            xgb_model.fit(X_train_scaled, y_train_np)
            
            rf_max_features = params['rf_max_features']
            if rf_max_features <= 0.33:
                rf_max_feat = 'sqrt'
            elif rf_max_features <= 0.66:
                rf_max_feat = 'log2'
            else:
                rf_max_feat = None
            
            rf_model = RandomForestRegressor(
                n_estimators=int(round(params['rf_n_estimators'])),
                max_depth=int(round(params['rf_max_depth'])) if params['rf_max_depth'] < 45 else None,
                min_samples_split=max(2, int(round(params['rf_min_samples_split']))),
                min_samples_leaf=max(1, int(round(params['rf_min_samples_leaf']))),
                max_features=rf_max_feat,
                random_state=random_state, n_jobs=-1
            )
            rf_model.fit(X_train_np, y_train_np)
            
            weight_xgb = float(params['weight_xgb'])
            pred_xgb = xgb_model.predict(X_test_scaled)
            pred_rf = rf_model.predict(X_test_np)
            y_pred = weight_xgb * pred_xgb + (1 - weight_xgb) * pred_rf
            
            y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=0.0, neginf=0.0)
            y_test_clean = np.nan_to_num(y_test_np, nan=0.0, posinf=0.0, neginf=0.0)
            
            rmse = np.sqrt(mean_squared_error(y_test_clean, y_pred))
            return rmse
            
        except Exception as e:
            print(f"集成模型训练过程中出错: {str(e)}")
            return float('inf')
    
    def data_prep_for_model(self, params):
        """执行五折交叉验证"""
        fold_results = [
            self.CV_train(params, getattr(self, f'X_train_{i}'), getattr(self, f'y_train_{i}'), 
                          getattr(self, f'X_test_{i}'), getattr(self, f'y_test_{i}'))
            for i in range(1, 6)
        ]
        
        valid_rmses = [rmse for rmse in fold_results if not np.isinf(rmse) and not np.isnan(rmse)]
        if len(valid_rmses) == 0: return -1
        
        rmse_mean = np.mean(valid_rmses)
        print(f"五折RMSE: {[round(r, 4) for r in fold_results]} | 平均RMSE: {rmse_mean:.4f}")
        return -1 * rmse_mean 
    
    def ensemble_evaluate(self, xgb_n_estimators, xgb_max_depth, xgb_learning_rate, xgb_subsample,
                          rf_n_estimators, rf_max_depth, rf_min_samples_split, rf_min_samples_leaf, rf_max_features,
                          weight_xgb):
        """集成模型超参数评估函数"""
        params = {
            'xgb_n_estimators': xgb_n_estimators, 'xgb_max_depth': xgb_max_depth,
            'xgb_learning_rate': xgb_learning_rate, 'xgb_subsample': xgb_subsample,
            'rf_n_estimators': rf_n_estimators, 'rf_max_depth': rf_max_depth,
            'rf_min_samples_split': rf_min_samples_split, 'rf_min_samples_leaf': rf_min_samples_leaf,
            'rf_max_features': rf_max_features,
            'weight_xgb': weight_xgb
        }
        return self.data_prep_for_model(params)
    
    def param(self):
        """执行贝叶斯优化超参数搜索"""
        pbounds = {
            'xgb_n_estimators': (50, 500),
            'xgb_max_depth': (3, 10),
            'xgb_learning_rate': (0.01, 0.3),
            'xgb_subsample': (0.6, 1.0),
            'rf_n_estimators': (50, 500),
            'rf_max_depth': (5, 50),
            'rf_min_samples_split': (2, 20),
            'rf_min_samples_leaf': (1, 10),
            'rf_max_features': (0, 1),
            'weight_xgb': (0.1, 0.9)
        }
        
        bo = BayesianOptimization(
            f=self.ensemble_evaluate,
            pbounds=pbounds,
            verbose=2,
            random_state=random_state
        )
        
        print('=' * 20)
        print('Starting Bayesian Optimization for XGBoost + RandomForest Ensemble...')
        print('=' * 20)
        
        try:
            bo.maximize(init_points=10, n_iter=500)
        except Exception as e:
            print(f"贝叶斯优化出错: {str(e)}")
            return {'xgb_n_estimators': 100, 'xgb_max_depth': 6, 'xgb_learning_rate': 0.1, 
                    'xgb_subsample': 0.8, 'rf_n_estimators': 100, 'rf_max_depth': 15,
                    'rf_min_samples_split': 5, 'rf_min_samples_leaf': 2, 'rf_max_features': 0.5,
                    'weight_xgb': 0.7}
        
        best_params = bo.max['params']
        print(f"最佳超参数: {best_params}")
        return best_params
    
    def final_model(self, X_train, y_train):
        """构建最终的集成模型"""
        try:
            params = self.param()
            
            X_train_np = np.nan_to_num(X_train[self.features].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            y_train_np = y_train.values.astype(np.float32)
            
            self.scaler_X.fit(X_train_np)
            X_train_scaled = self.scaler_X.transform(X_train_np)
            
            print("\n训练最终的 XGBoost 模型...")
            xgb_model = xgb.XGBRegressor(
                n_estimators=int(round(params['xgb_n_estimators'])),
                max_depth=int(round(params['xgb_max_depth'])),
                learning_rate=float(params['xgb_learning_rate']),
                subsample=float(params['xgb_subsample']),
                colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
                random_state=random_state, n_jobs=-1, verbosity=0
            )
            xgb_model.fit(X_train_scaled, y_train_np)
            
            print("训练最终的 RandomForest 模型...")
            rf_max_features = params['rf_max_features']
            if rf_max_features <= 0.33:
                rf_max_feat = 'sqrt'
            elif rf_max_features <= 0.66:
                rf_max_feat = 'log2'
            else:
                rf_max_feat = None
            
            rf_model = RandomForestRegressor(
                n_estimators=int(round(params['rf_n_estimators'])),
                max_depth=int(round(params['rf_max_depth'])) if params['rf_max_depth'] < 45 else None,
                min_samples_split=max(2, int(round(params['rf_min_samples_split']))),
                min_samples_leaf=max(1, int(round(params['rf_min_samples_leaf']))),
                max_features=rf_max_feat,
                random_state=random_state, n_jobs=-1
            )
            rf_model.fit(X_train_np, y_train_np)
            
            weight_xgb = float(params['weight_xgb'])
            print(f"模型融合完成！XGBoost权重: {weight_xgb:.2f}, RandomForest权重: {1-weight_xgb:.2f}")
            
            return EnsembleModel(xgb_model, rf_model, self.scaler_X, weight_xgb)
            
        except Exception as e:
            print(f"训练最终模型时出错: {str(e)}，使用默认参数...")
            xgb_m = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=random_state)
            rf_m = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=random_state, n_jobs=-1)
            xgb_m.fit(self.scaler_X.transform(X_train_np), y_train_np)
            rf_m.fit(X_train_np, y_train_np)
            return EnsembleModel(xgb_m, rf_m, self.scaler_X, 0.7)
    
    def result(self, model):
        """模型性能评估"""
        try:
            X_train_name = self.rem_name(self.X_train)
            X_test_name = self.rem_name(self.X_test)
            
            y_train, y_test = self.y_train, self.y_test
            
            X_train_np = np.nan_to_num(self.X_train[self.features].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            X_test_np = np.nan_to_num(self.X_test[self.features].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            
            y_train_pred = model.predict(X_train_np)
            y_pred = model.predict(X_test_np)
            
            y_train_pred = np.nan_to_num(y_train_pred, nan=0.0, posinf=0.0, neginf=0.0)
            y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=0.0, neginf=0.0)
            y_train_clean = np.nan_to_num(y_train.values, nan=0.0, posinf=0.0, neginf=0.0)
            y_test_clean = np.nan_to_num(y_test.values, nan=0.0, posinf=0.0, neginf=0.0)
            
            train_r2, train_rmse, train_mae = r2_score(y_train_clean, y_train_pred), np.sqrt(mean_squared_error(y_train_clean, y_train_pred)), mean_absolute_error(y_train_clean, y_train_pred)
            test_r2, test_rmse, test_mae = r2_score(y_test_clean, y_pred), np.sqrt(mean_squared_error(y_test_clean, y_pred)), mean_absolute_error(y_test_clean, y_pred)
            
            print("\n" + "=" * 60)
            print("XGBoost + RandomForest 集成模型性能结果")
            print("=" * 60)
            print(f"【训练集】 R²: {train_r2:.4f} | RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f}")
            print(f"【验证集】 R²: {test_r2:.4f} | RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f}")
            print("=" * 60)
            
            y_pred_df = pd.DataFrame(y_pred, columns=['pred'], index=self.X_test.index)
            y_test_df = pd.DataFrame(y_test_clean, columns=['Js'], index=self.X_test.index)
            y_train_pred_df = pd.DataFrame(y_train_pred, columns=['pred'], index=self.X_train.index)
            
            total = pd.concat([X_train_name, self.X_train[self.features], pd.Series(y_train_clean, name='Js', index=self.X_train.index)], axis=1)
            total_train = pd.concat([total, y_train_pred_df], axis=1)
            
            total_test = pd.concat([X_test_name, self.X_test[self.features], y_test_df, y_pred_df], axis=1)
            
            total_train['error'] = total_train['Js'] - total_train['pred']
            total_test['error'] = total_test['Js'] - total_test['pred']
            
            return self, total_train, total_test, total_train[['Js', 'pred', 'error']], total_test[['Js', 'pred', 'error']]
            
        except Exception as e:
            print(f"评估过程中出错: {str(e)}")
            return self, pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()


try:
    df = pd.read_excel('FO xgboost.xlsx', sheet_name='data')
    print("成功加载Excel文件")
except Exception as e:
    print(f"加载Excel文件失败: {str(e)}，使用示例数据...")
    np.random.seed(42)
    n_samples = 200
    df = pd.DataFrame({
        'ID': np.random.randint(1, 10, n_samples),
        'MPDC': np.random.normal(0, 1, n_samples), 'TMCC': np.random.normal(0, 1, n_samples),
        'MPDIPT': np.random.normal(0, 1, n_samples), 'TMCIPT': np.random.normal(0, 1, n_samples),
        'MT': np.random.normal(0, 1, n_samples), 'PTE': np.random.normal(0, 1, n_samples),
        'PTI': np.random.normal(0, 1, n_samples), 'OPS': np.random.normal(0, 1, n_samples),
        'WP': np.random.normal(0, 1, n_samples), 'MO': np.random.normal(0, 1, n_samples),
        'DST': np.random.normal(0, 1, n_samples), 'DSC': np.random.normal(0, 1, n_samples),
        'ST': np.random.normal(0, 1, n_samples), 'CV': np.random.normal(0, 1, n_samples),
        'Jw': np.random.normal(50, 20, n_samples), 'Js': np.random.normal(0, 1, n_samples),
        'R': np.random.normal(50, 20, n_samples), 'ref_ID': np.random.randint(1, 5, n_samples)
    })

expected_columns = ['ID', 'MPDC', 'TMCC', 'MPDIPT', 'TMCIPT', 'MT', 'PTE', 'PTI', 'OPS',
                  'CT', 'WCA', 'WP', 'MO', 'DST', 'DSC', 'ST', 'CV', 'Jw', 'Js', 'R', 'ref_ID']

if len(df.columns) == len(expected_columns): df.columns = expected_columns
df.index = range(len(df))

features = ['MPDC', 'TMCC', 'MPDIPT', 'TMCIPT', 'MT', 'PTE', 'PTI', 'OPS',
                  'CT', 'WCA', 'WP', 'MO', 'DST', 'DSC', 'ST', 'CV', ]
features = [col for col in features if col in df.columns]

if 'ref_ID' in df.columns:
    ref_id = pd.to_numeric(df['ref_ID'], errors='coerce')
    train, test = df[ref_id != 1].copy(), df[ref_id == 1].copy()
    if len(test) == 0:
        from sklearn.model_selection import train_test_split
        train, test = train_test_split(df, test_size=0.2, random_state=42)
else:
    from sklearn.model_selection import train_test_split
    train, test = train_test_split(df, test_size=0.2, random_state=42)

print(f"训练集大小: {len(train)}, 测试集大小: {len(test)}")
df = fill_nan_with_median(df, "全局数据集df")

try:
    one, two, three, four, five = CV_group(train.copy())
except Exception as e:
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    splits = list(kf.split(train))
    one, two, three, four, five = [train.iloc[splits[i][1]] for i in range(5)]

ml_model = ML_model(train, test, one, two, three, four, five, features)
final_model = ml_model.final_model(train, train['Js'])
result, total_train, total_test, train_error, test_error = ml_model.result(final_model)

try:
    import plotly.express as px
    import plotly.graph_objects as go
    
    if not total_train.empty and not total_test.empty:
        total_train['mark'] = 'train'
        total_test['mark'] = 'test'
        total_data = pd.concat([total_train, total_test])
    
        X_title, Y_title = 'Real R (%)', 'Pred R (%)'
        fig = px.scatter(total_data, x='Js', y='pred', color='mark', width=700, height=600,
                         labels={'Js': X_title, 'pred': Y_title},
                         color_discrete_map={'train': '1f77b4', 'test': 'ff7f0e'})
    
        fig.update_layout(
            xaxis=dict(title_text=X_title, title_font={"size": 25}, tickfont=dict(size=18), title_standoff=20, ticks="inside"),
            yaxis=dict(title_text=Y_title, title_font={"size": 25}, tickfont=dict(size=18), title_standoff=20, ticks="inside"),
            plot_bgcolor='rgb(256,256,256)', showlegend=True
        )
        fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
        fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
    
        min_val = min(total_data['Js'].min(), total_data['pred'].min())
        max_val = max(total_data['Js'].max(), total_data['pred'].max())
        fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val], 
                                 line=dict(color='red', width=2, dash='dash'), name='Perfect Prediction'))
    
        fig.update_traces(marker=dict(size=8, opacity=0.7, line=dict(width=0.5)), selector=dict(mode='markers'))
        fig.show()
        with pd.ExcelWriter('plot_data_output.xlsx', engine='openpyxl') as writer:
            total_train.to_excel(writer, sheet_name='train_data', index=False)
            total_test.to_excel(writer, sheet_name='test_data', index=False)
            total_data.to_excel(writer, sheet_name='all_plot_data', index=False)
            train_error.to_excel(writer, sheet_name='train_error', index=False)
            test_error.to_excel(writer, sheet_name='test_error', index=False)
        print("\n✅ 绘图数据已保存至 plot_data_outputJs.xlsx！")
        print("Sheet清单：train_data / test_data / all_plot_data / train_error / test_error")
except Exception as e:
    print(f"生成可视化图表出错: {str(e)}")

print("\n程序执行完成！")

In [ ]:
import joblib
import json
import os
import pandas as pd
import numpy as np

os.makedirs('saved_models', exist_ok=True)

joblib.dump(final_model, 'saved_models/model_Js.pkl')

features = ['MPDC', 'TMCC', 'MPDIPT', 'TMCIPT', 'MT', 'PTE', 'PTI', 'OPS', 
              'WCA', 'WP', 'MO', 'DST', 'DSC', 'ST' ]

bounds = {}
for feat in features:
    bounds[feat] = [float(df[feat].min()), float(df[feat].max())]

with open('saved_models/feature_bounds.json', 'w') as f:
    json.dump(bounds, f, indent=4)
    
print("模型和特征边界保存成功！")

In [ ]:
try:
    import shap
    
    print("\n" + "=" * 60)
    print("开始 SHAP 特征重要性分析...")
    print("=" * 60)
    
    X_test_np = np.nan_to_num(ml_model.X_test[features].values.astype(np.float32), 
                              nan=0.0, posinf=0.0, neginf=0.0)
    X_train_np = np.nan_to_num(ml_model.X_train[features].values.astype(np.float32), 
                               nan=0.0, posinf=0.0, neginf=0.0)
    
    print("\n[1/3] 计算 XGBoost SHAP 值...")
    xgb_explainer = shap.TreeExplainer(final_model.xgb_model)
    shap_values_xgb = xgb_explainer.shap_values(X_test_np)
    
    print("[2/3] 计算 RandomForest SHAP 值...")
    rf_explainer = shap.TreeExplainer(final_model.rf_model)
    shap_values_rf = rf_explainer.shap_values(X_test_np)
    if isinstance(shap_values_rf, list):
        shap_values_rf = shap_values_rf[0]
    
    print("[3/3] 按集成权重合并 SHAP 值...")
    weight_xgb = final_model.weight_xgb
    shap_values_ensemble = weight_xgb * shap_values_xgb + (1 - weight_xgb) * shap_values_rf
    
    ensemble_explanation = shap.Explanation(
        values=shap_values_ensemble,
        base_values=np.full(X_test_np.shape[0], 
                           weight_xgb * xgb_explainer.expected_value + 
                           (1 - weight_xgb) * rf_explainer.expected_value),
        data=X_test_np,
        feature_names=features
    )
    
    print("\n生成集成模型 SHAP 摘要图...")
    shap.summary_plot(ensemble_explanation, show=False, max_display=len(features))
    
    print("生成集成模型 SHAP 特征重要性条形图...")
    shap.plots.bar(ensemble_explanation, show=False, max_display=len(features))
    
    print("生成集成模型 SHAP 蜂群图...")
    shap.plots.beeswarm(ensemble_explanation, show=False, max_display=min(15, len(features)))
    
    print("\n生成 XGBoost 独立 SHAP 摘要图...")
    xgb_explanation = shap.Explanation(
        values=shap_values_xgb,
        base_values=xgb_explainer.expected_value,
        data=X_test_np,
        feature_names=features
    )
    shap.summary_plot(xgb_explanation, show=False, title="XGBoost SHAP Summary", max_display=len(features))
    
    print("生成 RandomForest 独立 SHAP 摘要图...")
    rf_explanation = shap.Explanation(
        values=shap_values_rf,
        base_values=rf_explainer.expected_value,
        data=X_test_np,
        feature_names=features
    )
    shap.summary_plot(rf_explanation, show=False, title="RandomForest SHAP Summary", max_display=len(features))
    
    mean_abs_shap = np.abs(shap_values_ensemble).mean(axis=0)
    shap_importance_df = pd.DataFrame({
        'Feature': features,
        'Mean_|SHAP|': mean_abs_shap,
        'Rank': range(1, len(features) + 1)
    }).sort_values('Mean_|SHAP|', ascending=False).reset_index(drop=True)
    shap_importance_df['Rank'] = range(1, len(shap_importance_df) + 1)
    
    print("\n" + "-" * 40)
    print("集成模型 SHAP 特征重要性排名:")
    print("-" * 40)
    print(shap_importance_df.to_string(index=False))
    print("-" * 40)
    
    print("\n✅ SHAP 分析完成！")
    
except ImportError:
    print("\n⚠️ 未安装 shap 库，跳过 SHAP 分析。请运行: pip install shap")
except Exception as e:
    print(f"\n⚠️ SHAP 分析过程中出错: {str(e)}")
    import traceback
    traceback.print_exc()

In [ ]:
try:
    import plotly.express as px
    import plotly.graph_objects as go
    
    if not total_train.empty and not total_test.empty:
        total_train['mark'] = 'train'
        total_test['mark'] = 'test'
        total_data = pd.concat([total_train, total_test])
    
        X_title, Y_title = 'Real R (%)', 'Pred R (%)'
        fig = px.scatter(total_data, x='Js', y='pred', color='mark', width=700, height=600,
                         labels={'Js': X_title, 'pred': Y_title},
                         color_discrete_map={'train': '1f77b4', 'test': 'ff7f0e'})
    
        fig.update_layout(
            xaxis=dict(title_text=X_title, title_font={"size": 25}, tickfont=dict(size=18), title_standoff=20, ticks="inside"),
            yaxis=dict(title_text=Y_title, title_font={"size": 25}, tickfont=dict(size=18), title_standoff=20, ticks="inside"),
            plot_bgcolor='rgb(256,256,256)', showlegend=True
        )
        fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
        fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
    
        min_val = min(total_data['Js'].min(), total_data['pred'].min())
        max_val = max(total_data['Js'].max(), total_data['pred'].max())
        fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val], 
                                 line=dict(color='red', width=2, dash='dash'), name='Perfect Prediction'))
    
        fig.update_traces(marker=dict(size=8, opacity=0.7, line=dict(width=0.5)), selector=dict(mode='markers'))
        fig.show()
        with pd.ExcelWriter('plot_data_output.xlsx', engine='openpyxl') as writer:
            total_train.to_excel(writer, sheet_name='train_data', index=False)
            total_test.to_excel(writer, sheet_name='test_data', index=False)
            total_data.to_excel(writer, sheet_name='all_plot_data', index=False)
            train_error.to_excel(writer, sheet_name='train_error', index=False)
            test_error.to_excel(writer, sheet_name='test_error', index=False)
        print("\n✅ 绘图数据已保存至 plot_data_outputJs.xlsx！")
        print("Sheet清单：train_data / test_data / all_plot_data / train_error / test_error")
except Exception as e:
    print(f"生成可视化图表出错: {str(e)}")

print("\n程序执行完成！")